In [ ]:
# @title Setup
import json, os, sys, time
for path in ("..", "/content/mnist"):  # project root on this computer, uploaded folder on Colab
    if os.path.isfile(os.path.join(path, "models.py")) and os.path.abspath(path) not in sys.path:
        sys.path.insert(0, os.path.abspath(path))
import torch
from torchvision import datasets, transforms
import models, quicktrain

ON_COLAB = os.path.isdir("/content")
DATA_DIR = "/content/data" if ON_COLAB else "../data"
SUBSET = int(os.environ.get("MNIST_SUBSET", "5000" if ON_COLAB else "500"))
STEPS = int(os.environ.get("MNIST_STEPS", "300" if ON_COLAB else "30"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device, "| torch", torch.__version__, "| subset", SUBSET, "| steps", STEPS)

In [ ]:
# @title Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_full = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
train_loader = quicktrain.subset_loader(train_full, SUBSET, 64, seed=0)
test_loader = quicktrain.subset_loader(test_full, 2000, 500, seed=1, shuffle=False)
print("train items", len(train_loader.dataset), "| test items", len(test_loader.dataset))

In [ ]:
# @title Train both models
results = {}
for name in models.MODEL_NAMES:
    torch.manual_seed(0)
    model = models.build_model(name).to(device)
    started = time.time()
    losses = quicktrain.train_steps(model, train_loader, STEPS, device)
    results[name] = {
        "final_loss": round(losses[-1], 4),
        "test_accuracy": round(quicktrain.accuracy(model, test_loader, device), 4),
        "seconds": round(time.time() - started, 1),
    }
    print(name, results[name])

In [ ]:
# @title Extra model from a second script
import importlib, models_extra
importlib.reload(models_extra)  # picks up an edited copy without restarting the kernel
torch.manual_seed(0)
wide = models_extra.WideMLP().to(device)
started = time.time()
losses = quicktrain.train_steps(wide, train_loader, STEPS, device)
results["wide_mlp"] = {
    "final_loss": round(losses[-1], 4),
    "test_accuracy": round(quicktrain.accuracy(wide, test_loader, device), 4),
    "seconds": round(time.time() - started, 1),
}
print("wide_mlp", results["wide_mlp"])

In [ ]:
# @title Compare
best = max(results, key=lambda name: results[name]["test_accuracy"])
print("best model:", best, "with test accuracy", results[best]["test_accuracy"])

In [ ]:
# @title Save results
out = "/content/notebook_results.json" if ON_COLAB else "notebook_results.json"
with open(out, "w") as handle:
    json.dump({"device": str(device), "subset": SUBSET, "steps": STEPS, "results": results}, handle, indent=2)
print("saved", out)